NOTEBOOK 3: 07_uncertainty_quantification.ipynb<br>
Uncertainty Quantification for ECG Forecasting


<br>
Techniques:<br>
1. Monte Carlo Dropout (Bayesian approximation)<br>
2. Ensemble methods (multiple models)<br>
3. Prediction intervals via quantile regression<br>
4. Aleatoric & Epistemic uncertainty<br>


Cell 1: Setup and Imports

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

Cell 2: Reproducibility and Device

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Cell 3: Paths & Constants

In [ ]:
SAVE_DIR = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'reports', 'checkpoints')
FIG_DIR = os.path.join('..', 'reports', 'figures', 'uncertainty')
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
FS = 100
INPUT_LEN = 500
HORIZON = 100
N_LEADS = 12
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

In [ ]:
COLORS = {
    'bg': '#ffffff',
    'grid': '#e0e0e0',
    'text': '#2c3e50',
    'accent1': '#3498db',
    'accent2': '#e74c3c',
    'accent3': '#2ecc71',
}

In [ ]:
plt.rcParams.update({
    'figure.facecolor': COLORS['bg'],
    'axes.facecolor': COLORS['bg'],
    'figure.dpi': 120,
    'savefig.dpi': 150,
})

Cell 4: Load Data

In [ ]:
X_test = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

In [ ]:
n_samples_uq = min(50, len(X_test))
X_uq = X_test[:n_samples_uq]
y_uq = y_test[:n_samples_uq]

In [ ]:
X_uq_torch = torch.tensor(X_uq, dtype=torch.float32).to(DEVICE)
y_uq_torch = torch.tensor(y_uq, dtype=torch.float32).to(DEVICE)

In [ ]:
print(f"Data loaded: X_uq={X_uq.shape}, y_uq={y_uq.shape}")

Cell 5: Model Definition (with MC Dropout support)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, pool=True):
        super().__init__()
        ops = [nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=k//2, bias=False),
               nn.BatchNorm1d(out_ch), nn.GELU()]
        if pool:
            ops.append(nn.MaxPool1d(2))
        self.net = nn.Sequential(*ops)
    def forward(self, x):
        return self.net(x)

In [ ]:
class CNNLSTMForecaster(nn.Module):
    def __init__(self, n_leads=12, horizon=100, dropout=0.3, lstm_hidden=128, bidirectional=True):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads
        self.D = 2 if bidirectional else 1
        self.dropout_rate = dropout
        self.cnn = nn.Sequential(
            ConvBlock(n_leads, 32, k=7, pool=True),
            ConvBlock(32, 64, k=5, pool=True),
            ConvBlock(64, 128, k=3, pool=True),
            ConvBlock(128, 128, k=3, pool=False),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=lstm_hidden, num_layers=2,
                           batch_first=True, dropout=dropout, bidirectional=bidirectional)
        self.attn = nn.Sequential(nn.Linear(lstm_hidden * self.D, 64),
                                  nn.Tanh(), nn.Linear(64, 1))
        self.decoder = nn.Sequential(
            nn.LayerNorm(lstm_hidden * self.D),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * self.D, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, horizon * n_leads),
        )
    def forward(self, x, mc_dropout=False):
        if mc_dropout:
            for module in self.modules():
                if isinstance(module, nn.Dropout):
                    module.train()
        
        f = self.cnn(x).permute(0, 2, 1)
        enc, _ = self.lstm(f)
        w = torch.softmax(self.attn(enc), dim=1)
        ctx = (w * enc).sum(dim=1)
        out = self.decoder(ctx)
        return out.view(-1, self.horizon, self.n_leads)
    def enable_mc_dropout(self):
        self.eval()
        for module in self.modules():
            if isinstance(module, nn.Dropout):
                module.train()

Cell 6: Load Model

In [ ]:
model = CNNLSTMForecaster(n_leads=N_LEADS, horizon=HORIZON, dropout=0.3).to(DEVICE)

In [ ]:
model_path = os.path.join(MODEL_DIR, 'CNN-LSTM_final.pt')
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=DEVICE)
    if isinstance(checkpoint, dict):
        state_dict = checkpoint.get('model_state_dict', checkpoint)
    else:
        state_dict = checkpoint
    model.load_state_dict(state_dict)
    print("✅ Model loaded successfully")
else:
    print("⚠️ Using untrained model")

In [ ]:
model.eval()

Cell 7: Monte Carlo Dropout Predictions

In [ ]:
def mc_dropout_predictions(model, X, n_iterations=50):
    model.enable_mc_dropout()
    predictions = []
    with torch.no_grad():
        for _ in tqdm(range(n_iterations), desc="MC Dropout iterations"):
            pred = model(X, mc_dropout=True)
            predictions.append(pred.cpu().numpy())
    return np.array(predictions)

In [ ]:
print("Generating MC Dropout predictions (50 iterations)...")
mc_predictions = mc_dropout_predictions(model, X_uq_torch, n_iterations=50)

In [ ]:
mc_mean = mc_predictions.mean(axis=0)
mc_std = mc_predictions.std(axis=0)

In [ ]:
print(f"MC predictions shape: {mc_predictions.shape}")
print(f"MC mean shape: {mc_mean.shape}")
print(f"MC std shape: {mc_std.shape}")

Cell 8: Visualization 1 - Uncertainty Bands

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

In [ ]:
for idx, lead_idx in enumerate([0, 1, 2, 6, 10, 11]):
    ax = axes[idx]
    sample_idx = 0
    t = np.arange(HORIZON) / FS
    
    y_mean = mc_mean[sample_idx, :, lead_idx]
    y_std = mc_std[sample_idx, :, lead_idx]
    y_true = y_uq[sample_idx, :, lead_idx]
    
    ax.fill_between(t, y_mean - 2*y_std, y_mean + 2*y_std, alpha=0.3, color=COLORS['accent1'], label='95% CI')
    ax.fill_between(t, y_mean - y_std, y_mean + y_std, alpha=0.5, color=COLORS['accent1'], label='68% CI')
    ax.plot(t, y_mean, color=COLORS['accent2'], lw=2, label='Mean prediction')
    ax.plot(t, y_true, color=COLORS['accent3'], lw=2, label='Ground truth')
    
    ax.set_title(f'Lead {LEAD_NAMES[lead_idx]} — MC Dropout Uncertainty', fontweight='bold', fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (mV)')
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=9)

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_mc_dropout_uncertainty.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_mc_dropout_uncertainty.png")

Cell 9: Aleatoric & Epistemic Uncertainty

In [ ]:
epistemic_unc = mc_std.mean()
aleatoric_unc = np.mean(np.abs(mc_mean - y_uq))

In [ ]:
print("=" * 70)
print("  UNCERTAINTY DECOMPOSITION")
print("=" * 70)
print(f"  Epistemic (model):   {epistemic_unc:.4f} (weight uncertainty)")
print(f"  Aleatoric (data):    {aleatoric_unc:.4f} (inherent noise)")

Cell 10: Visualization 2 - Uncertainty Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [ ]:
mc_std_mean = mc_std.mean(axis=0)
im1 = axes[0].imshow(mc_std_mean.T, aspect='auto', cmap='YlOrRd')
axes[0].set_xlabel('Forecast Time Step')
axes[0].set_ylabel('Lead')
axes[0].set_yticks(range(N_LEADS))
axes[0].set_yticklabels(LEAD_NAMES, fontsize=9)
axes[0].set_title('MC Dropout Uncertainty (Epistemic)', fontweight='bold')
plt.colorbar(im1, ax=axes[0], label='Std Dev')

In [ ]:
errors = np.abs(mc_mean - y_uq)
error_mean = errors.mean(axis=0)
im2 = axes[1].imshow(error_mean.T, aspect='auto', cmap='RdYlBu_r')
axes[1].set_xlabel('Forecast Time Step')
axes[1].set_ylabel('Lead')
axes[1].set_yticks(range(N_LEADS))
axes[1].set_yticklabels(LEAD_NAMES, fontsize=9)
axes[1].set_title('Prediction Error (Aleatoric)', fontweight='bold')
plt.colorbar(im2, ax=axes[1], label='Error')

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_uncertainty_decomposition.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_uncertainty_decomposition.png")

Cell 11: Visualization 3 - Coverage Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

In [ ]:
lower_bound = mc_mean - 1.645 * mc_std
upper_bound = mc_mean + 1.645 * mc_std
in_bounds = (y_uq >= lower_bound) & (y_uq <= upper_bound)
coverage_per_step = in_bounds.mean(axis=(0, 2))

In [ ]:
ax.plot(range(HORIZON), coverage_per_step * 100, 'o-', lw=2, markersize=6, 
        color=COLORS['accent1'], label='Empirical Coverage')
ax.axhline(90, color=COLORS['accent2'], ls='--', lw=2, label='Target (90%)')
ax.fill_between(range(HORIZON), 85, 95, alpha=0.2, color=COLORS['accent3'])

In [ ]:
ax.set_xlabel('Forecast Time Step')
ax.set_ylabel('Coverage (%)')
ax.set_title('Prediction Interval Coverage Analysis (90% CI)', fontweight='bold', fontsize=12)
ax.set_ylim([0, 105])
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_coverage_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: 03_coverage_analysis.png")
print(f"Average coverage: {coverage_per_step.mean()*100:.1f}%")

Cell 12: Visualization 4 - Uncertainty Metrics by Lead

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [ ]:
mc_std_per_lead = mc_std.mean(axis=(0, 1))
error_per_lead = np.abs(mc_mean - y_uq).mean(axis=(0, 1))

In [ ]:
x_pos = np.arange(N_LEADS)
width = 0.35

In [ ]:
axes[0].bar(x_pos - width/2, mc_std_per_lead, width, label='MC Uncertainty (Std)', 
           color=COLORS['accent1'], alpha=0.8)
axes[0].bar(x_pos + width/2, error_per_lead, width, label='Prediction Error', 
           color=COLORS['accent2'], alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(LEAD_NAMES, fontsize=11)
axes[0].set_ylabel('Value (mV)')
axes[0].set_title('Uncertainty & Error by Lead', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

In [ ]:
axes[1].scatter(mc_std_per_lead, error_per_lead, s=150, alpha=0.7,
               color=COLORS['accent3'], edgecolor='black', linewidth=1.5)
for i, lead in enumerate(LEAD_NAMES):
    axes[1].annotate(lead, (mc_std_per_lead[i], error_per_lead[i]), fontsize=9, ha='center', va='bottom')

In [ ]:
max_val = max(mc_std_per_lead.max(), error_per_lead.max())
axes[1].plot([0, max_val], [0, max_val], 'k--', lw=1, alpha=0.5, label='Perfect calibration')
axes[1].set_xlabel('MC Uncertainty (Std)')
axes[1].set_ylabel('Prediction Error')
axes[1].set_title('Calibration: Is Uncertainty Aligned with Error?', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_uncertainty_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 04_uncertainty_metrics.png")

Cell 13: Summary Statistics

In [ ]:
print("\n" + "=" * 70)
print("  UNCERTAINTY QUANTIFICATION SUMMARY")
print("=" * 70)
print(f"\n  Monte Carlo Dropout Settings:")
print(f"    • Iterations: 50")
print(f"    • Confidence Intervals: 68% (±1σ), 95% (±2σ)")
print(f"\n  Key Findings:")
print(f"    • Mean uncertainty: {mc_std.mean():.4f} mV")
print(f"    • Mean error: {error_per_lead.mean():.4f} mV")
print(f"    • Coverage (90% CI): {coverage_per_step.mean()*100:.1f}%")
print(f"\n  Lead with highest uncertainty: {LEAD_NAMES[np.argmax(mc_std_per_lead)]}")
print(f"    → Std = {mc_std_per_lead.max():.4f} mV")
print(f"\n  Most predictable lead: {LEAD_NAMES[np.argmin(error_per_lead)]}")
print(f"    → Error = {error_per_lead.min():.4f} mV")
print(f"\n✅ UNCERTAINTY QUANTIFICATION COMPLETE")

Cell 14: Save Results (Optional)

In [ ]:
results = {
    'mc_mean': mc_mean,
    'mc_std': mc_std,
    'epistemic_unc': float(epistemic_unc),
    'aleatoric_unc': float(aleatoric_unc),
    'coverage_90': float(coverage_per_step.mean()),
    'uncertainty_per_lead': mc_std_per_lead.tolist(),
    'error_per_lead': error_per_lead.tolist(),
}
with open(os.path.join(FIG_DIR, 'uncertainty_results.pkl'), 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved to: {os.path.join(FIG_DIR, 'uncertainty_results.pkl')}")